# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vincentoei/flyrank-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason code

**Rule in plain words:** A page is a growth candidate if it already has real search volume (at least 500 impressions in the last 90 days) *and* a large share of that volume arrived recently. Recent concentration is a simple momentum signal: demand is moving toward the page now, not months ago.

**Score:** `recent_share * log1p(gsc_impressions_90d)` when `gsc_impressions_90d >= 500`, otherwise `0`.

**Reason code:** `momentum_with_volume`.

**Action label:** `expand_and_protect` for scored pages, `monitor` for everyone else.

This rule leans on two signals we will check first: volume (flag-linked to FlyRank's quick-win / visible-page logic) and recent impression concentration (momentum).

In [1]:
%pip install -q duckdb pandas scikit-learn matplotlib

import os
import json
import getpass
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import duckdb

# Load .env so HF_TOKEN is available without a prompt
env_path = Path.cwd().parents[1] / '.env'
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            key, value = line.split('=', 1)
            os.environ.setdefault(key, value)

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Helpers
def spearman_rank_corr(x, y):
    xr = pd.Series(x).rank()
    yr = pd.Series(y).rank()
    return float(np.corrcoef(xr, yr)[0, 1])

def verdict_numeric(table, base_rate, floor=50):
    rates = table['growth_rate'].tolist()
    ns = table['n'].tolist()
    if any(n < floor for n in ns):
        return 'MIXED — at least one bucket below sample-size floor.'
    if all(rates[i] <= rates[i + 1] for i in range(len(rates) - 1)) and rates[-1] > base_rate:
        return 'CONFIRMED — growth rate rises with the signal.'
    if all(rates[i] >= rates[i + 1] for i in range(len(rates) - 1)) and rates[0] > base_rate:
        return 'OPPOSITE — growth rate falls as the signal rises.'
    return 'MIXED — no clear monotonic pattern.'

def precision_at_k(scores, labels, k=10):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

### Load the honest feature vector

Same windows and filters as `w03_feature_leakage_check.ipynb`. The label is the only future-looking column.

In [3]:
token = os.getenv('HF_TOKEN')
if not token:
    token = getpass.getpass('Hugging Face token: ')

conn = duckdb.connect()
conn.execute('INSTALL httpfs;')
conn.execute('LOAD httpfs;')
conn.execute(f'''
    CREATE OR REPLACE SECRET hf_token (
        TYPE HTTP,
        EXTRA_HTTP_HEADERS MAP {{
            'Authorization': 'Bearer {token}'
        }}
    );
''')

conn.execute('''
    CREATE OR REPLACE VIEW dim_content AS
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
''')

conn.execute('''
    CREATE OR REPLACE VIEW dim_clients AS
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')
''')

fv_query = '''
    WITH feature_window AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS gsc_impressions_90d,
            AVG(gsc_avg_position) AS gsc_avg_position_90d,
            SUM(gsc_clicks) AS gsc_clicks_90d,
            CASE WHEN SUM(gsc_impressions) > 0 THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions) ELSE 0 END AS gsc_ctr_90d,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN sessions_organic ELSE 0 END) AS sessions_organic_90d,
            COUNT(DISTINCT report_date) AS days_with_impressions_90d
        FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-0[1-3]/*.parquet')
        WHERE report_date BETWEEN '2026-01-01' AND '2026-03-31'
        GROUP BY client_hash_id, content_hash_id
    ),
    last_30 AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS gsc_impressions_last_30d
        FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
        GROUP BY client_hash_id, content_hash_id
    ),
    label_window AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS gsc_impressions_next_30d
        FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet')
        WHERE report_date BETWEEN '2026-04-01' AND '2026-04-30'
        GROUP BY client_hash_id, content_hash_id
    ),
    decision_date AS (
        SELECT DATE '2026-03-31' AS d
    )
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.gsc_impressions_90d,
        f.gsc_avg_position_90d,
        f.gsc_clicks_90d,
        f.gsc_ctr_90d,
        f.sessions_organic_90d,
        f.days_with_impressions_90d,
        l30.gsc_impressions_last_30d,
        l.gsc_impressions_next_30d,
        dim_content.search_volume,
        dim_content.competition,
        dim_content.cpc,
        dim_content.word_count,
        dim_content.content_type,
        dim_content.main_intent,
        (d.d - dim_content.content_created_date)::INTEGER AS content_age_days,
        CASE
            WHEN l.gsc_impressions_next_30d >= 1.20 * l30.gsc_impressions_last_30d
                 AND l30.gsc_impressions_last_30d >= 100
            THEN 1 ELSE 0
        END AS growth_label
    FROM feature_window f
    LEFT JOIN last_30 l30 ON f.content_hash_id = l30.content_hash_id
    LEFT JOIN label_window l ON f.content_hash_id = l.content_hash_id
    LEFT JOIN dim_content ON f.content_hash_id = dim_content.content_hash_id
    CROSS JOIN decision_date d
    WHERE f.gsc_impressions_90d >= 300
      AND dim_content.content_created_date <= d.d
      AND dim_content.word_count IS NOT NULL
'''

feature_vector = conn.execute(fv_query).df()
print(f'Rows: {len(feature_vector):,}')
print(f'Base growth rate: {feature_vector["growth_label"].mean():.3%}')

window_check = conn.execute('''
    SELECT MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-0[1-3]/*.parquet')
    WHERE report_date BETWEEN '2026-01-01' AND '2026-03-31'
''').df()
print('\nFeature window span:')
print(window_check.to_string(index=False))
assert window_check['max_date'].iloc[0] <= pd.Timestamp('2026-03-31'), 'Feature window leaks into the target month!'

Rows: 67,478
Base growth rate: 24.731%



Feature window span:
  min_date   max_date
2026-01-01 2026-03-31


### Signal check #1 — Volume (flag-linked to quick-win / visible-page logic)

**Claim:** Pages with higher 90-day impression volume are more likely to grow.

**Test:** Bucket pages by `gsc_impressions_90d` and compare growth rates.

In [4]:
volume_bins = [300, 500, 1000, 3000, 10000, np.inf]
volume_labels = ['300-500', '500-1k', '1k-3k', '3k-10k', '10k+']

df_vol = feature_vector.copy()
df_vol['volume_tier'] = pd.cut(
    df_vol['gsc_impressions_90d'], bins=volume_bins, labels=volume_labels, right=False, include_lowest=True
)
base_rate_vol = df_vol['growth_label'].mean()

table_vol = df_vol.groupby('volume_tier', observed=True).agg(
    n=('growth_label', 'size'),
    growth_rate=('growth_label', 'mean')
).reset_index()
table_vol['lift_over_base'] = table_vol['growth_rate'] - base_rate_vol

print(f'Base growth rate: {base_rate_vol:.3%}')
print(table_vol.to_string(index=False))

verdict_vol = verdict_numeric(table_vol, base_rate_vol)
print(f'\nVerdict: {verdict_vol}')

Base growth rate: 24.731%
volume_tier     n  growth_rate  lift_over_base
    300-500  8159     0.273440        0.026130
     500-1k 11447     0.271600        0.024289
      1k-3k 17870     0.267487        0.020177
     3k-10k 17012     0.218610       -0.028700
       10k+ 12990     0.219323       -0.027988

Verdict: MIXED — no clear monotonic pattern.


### Signal check #2 — Recent impression concentration (momentum)

**Claim:** Pages with a larger share of their 90-day impressions in the last 30 days are more likely to grow.

**Test:** Bucket pages into quintiles of `recent_share = gsc_impressions_last_30d / gsc_impressions_90d`.

In [5]:
df_mom = feature_vector.dropna(subset=['gsc_impressions_last_30d', 'gsc_impressions_90d']).copy()
df_mom['recent_share'] = (df_mom['gsc_impressions_last_30d'] / df_mom['gsc_impressions_90d']).clip(0, 1)
df_mom['momentum_quintile'] = pd.qcut(df_mom['recent_share'], q=5, duplicates='drop')

base_rate_mom = df_mom['growth_label'].mean()

table_mom = df_mom.groupby('momentum_quintile', observed=True).agg(
    n=('growth_label', 'size'),
    growth_rate=('growth_label', 'mean'),
    median_recent_share=('recent_share', 'median')
).reset_index()
table_mom['lift_over_base'] = table_mom['growth_rate'] - base_rate_mom

print(f'Base growth rate: {base_rate_mom:.3%}')
print(table_mom.to_string(index=False))

rho = spearman_rank_corr(df_mom['recent_share'], df_mom['growth_label'])
print(f'\nSpearman rank correlation with growth: {rho:.3f}')

verdict_mom = verdict_numeric(table_mom, base_rate_mom)
print(f'\nVerdict: {verdict_mom}')

Base growth rate: 25.891%
momentum_quintile     n  growth_rate  median_recent_share  lift_over_base
  (-0.001, 0.346] 12892     0.233401             0.293266       -0.025505
    (0.346, 0.45] 12891     0.165387             0.396307       -0.093519
    (0.45, 0.608] 12891     0.177721             0.515625       -0.081184
   (0.608, 0.884] 12891     0.233574             0.731384       -0.025331
     (0.884, 1.0] 12891     0.484447             1.000000        0.225541

Spearman rank correlation with growth: 0.196

Verdict: MIXED — no clear monotonic pattern.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
df = feature_vector.copy()

# Pre-decision momentum signal
df['recent_share'] = (df['gsc_impressions_last_30d'] / df['gsc_impressions_90d']).clip(0, 1).fillna(0)

# Baseline score: volume + momentum, no fitted weights
df['baseline_score'] = np.where(
    df['gsc_impressions_90d'] >= 500,
    df['recent_share'] * np.log1p(df['gsc_impressions_90d']),
    0.0
)

# One reason code, one action label
df['reason_code'] = np.where(df['baseline_score'] > 0, 'momentum_with_volume', 'none')
df['action_label'] = np.where(df['baseline_score'] > 0, 'expand_and_protect', 'monitor')

df['baseline_rank'] = df['baseline_score'].rank(method='first', ascending=False).astype(int)

# Output columns
out_cols = [
    'client_hash_id', 'content_hash_id', 'baseline_rank', 'baseline_score',
    'reason_code', 'action_label',
    'gsc_impressions_90d', 'gsc_impressions_last_30d', 'recent_share',
    'gsc_avg_position_90d', 'gsc_ctr_90d', 'word_count', 'content_age_days',
    'days_with_impressions_90d', 'main_intent', 'growth_label'
]
queue = df[out_cols].sort_values('baseline_rank')

output_path = Path.cwd().parent / 'outputs' / 'baseline_action_score.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(output_path, index=False)

# Metrics
base_rate = df['growth_label'].mean()
p10 = precision_at_k(df['baseline_score'], df['growth_label'], k=10)
p50 = precision_at_k(df['baseline_score'], df['growth_label'], k=50)

print(f'Rows in queue: {len(queue):,}')
print(f'Pages flagged (score > 0): {(df["baseline_score"] > 0).sum():,}')
print(f'Base growth rate: {base_rate:.3%}')
print(f'Precision@10: {p10:.3f}')
print(f'Precision@50: {p50:.3f}')
print(f'CSV written to: {output_path}')

metrics = {
    'rows': int(len(queue)),
    'flagged_count': int((df['baseline_score'] > 0).sum()),
    'base_growth_rate': float(base_rate),
    'precision_at_10': float(p10),
    'precision_at_50': float(p50),
    'score_features': ['gsc_impressions_90d', 'gsc_impressions_last_30d'],
    'reason_code': 'momentum_with_volume',
    'action_label': 'expand_and_protect',
    'volume_verdict': verdict_vol,
    'momentum_verdict': verdict_mom,
    'timestamp': datetime.now(timezone.utc).isoformat()
}
metrics_path = output_path.with_name('baseline_action_score_metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'Metrics written to: {metrics_path}')

print('\nTop 5 rows:')
print(queue.head(5).to_string(index=False))

Rows in queue: 67,478
Pages flagged (score > 0): 57,152
Base growth rate: 24.731%
Precision@10: 0.300
Precision@50: 0.480
CSV written to: /home/vincentoei/projects/flyrank-ml-internship-starter/work/outputs/baseline_action_score.csv
Metrics written to: /home/vincentoei/projects/flyrank-ml-internship-starter/work/outputs/baseline_action_score_metrics.json

Top 5 rows:
         client_hash_id          content_hash_id  baseline_rank  baseline_score          reason_code       action_label  gsc_impressions_90d  gsc_impressions_last_30d  recent_share  gsc_avg_position_90d  gsc_ctr_90d  word_count  content_age_days  days_with_impressions_90d   main_intent  growth_label
client_9958f0a7ae1df715 content_cd3d932d4e1c8db0              1       11.337388 momentum_with_volume expand_and_protect              89874.0                   89332.0      0.993969              7.845988     0.000056        3222               447                         90 informational             0
client_a80fca3f171ed1de cont

## 3. Top-10 review

*For each of the top 10: action, reason code, confidence note, and what would make it wrong.*

In [7]:
def review_note(row):
    return (
        f"Action: {row['action_label']} | "
        f"Why: {row['gsc_impressions_90d']:,.0f} 90d impressions, "
        f"{row['recent_share']:.1%} in the last 30d | "
        f"Wrong if: the last-30d spike was a one-off event or seasonality, not sustained demand."
    )

top10 = queue.head(10).copy()
top10['review_note'] = top10.apply(review_note, axis=1)

print('TOP-10 REVIEW')
print('=' * 80)
for _, row in top10.iterrows():
    print(f"{row['baseline_rank']:>2}. {row['review_note']}")

# Also show as a compact table
print('\nCompact table:')
print(top10[['baseline_rank', 'content_hash_id', 'baseline_score', 'reason_code', 'action_label', 'review_note']].to_string(index=False))

TOP-10 REVIEW
 1. Action: expand_and_protect | Why: 89,874 90d impressions, 99.4% in the last 30d | Wrong if: the last-30d spike was a one-off event or seasonality, not sustained demand.
 2. Action: expand_and_protect | Why: 83,788 90d impressions, 100.0% in the last 30d | Wrong if: the last-30d spike was a one-off event or seasonality, not sustained demand.
 3. Action: expand_and_protect | Why: 91,150 90d impressions, 98.7% in the last 30d | Wrong if: the last-30d spike was a one-off event or seasonality, not sustained demand.
 4. Action: expand_and_protect | Why: 82,991 90d impressions, 99.3% in the last 30d | Wrong if: the last-30d spike was a one-off event or seasonality, not sustained demand.
 5. Action: expand_and_protect | Why: 108,584 90d impressions, 96.1% in the last 30d | Wrong if: the last-30d spike was a one-off event or seasonality, not sustained demand.
 6. Action: expand_and_protect | Why: 63,257 90d impressions, 99.9% in the last 30d | Wrong if: the last-30d spike was 

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [8]:
# Weak picks: pages that scored only because of huge volume, with below-median momentum
flagged = df[df['baseline_score'] > 0].copy()
median_momentum = flagged['recent_share'].median()
weak = flagged[flagged['recent_share'] < median_momentum].sort_values('baseline_rank').head(5)

print(f'Median recent_share among flagged pages: {median_momentum:.3f}')
print('\nWeak picks (high score but weak momentum):')
print(weak[['baseline_rank', 'content_hash_id', 'gsc_impressions_90d', 'gsc_impressions_last_30d',
            'recent_share', 'baseline_score', 'action_label']].to_string(index=False))

# Leakage check
features_used = ['gsc_impressions_90d', 'gsc_impressions_last_30d']
excluded_fields = {
    'gsc_impressions_next_30d', 'trend_direction', 'trend_pct',
    'health_score', 'priority_score', 'action_type',
    'content_updated_date', 'last_optimized_date'
}
violations = set(features_used) & excluded_fields

print('\nLEAKAGE CHECK')
print('=' * 60)
print(f'Features used in the score: {features_used}')
print(f'Excluded fields checked: {sorted(excluded_fields)}')
print(f'Violations: {len(violations)}')
assert len(violations) == 0, f'Leakage risk: {violations}'
assert window_check['max_date'].iloc[0] <= pd.Timestamp('2026-03-31'), 'Feature window leaks into April!'
print('No leakage detected.')

Median recent_share among flagged pages: 0.499

Weak picks (high score but weak momentum):
 baseline_rank          content_hash_id  gsc_impressions_90d  gsc_impressions_last_30d  recent_share  baseline_score       action_label
         13022 content_e8a52cf3d5988c07             508139.0                  244931.0      0.482016        6.332970 expand_and_protect
         13091 content_ec2e0346994fb5a5             509532.0                  245276.0      0.481375        6.325870 expand_and_protect
         15210 content_77276ad7a26f4905             241683.0                  116707.0      0.482893        5.985644 expand_and_protect
         15374 content_fe3fd3422852d721             261114.0                  124720.0      0.477646        5.957540 expand_and_protect
         16002 content_93aadc6eeccba7b8             146256.0                   71709.0      0.490298        5.831171 expand_and_protect

LEAKAGE CHECK
Features used in the score: ['gsc_impressions_90d', 'gsc_impressions_last_30d'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.